# Consolidación Survey123 → capas finales

Este notebook consolida los atributos de la **tabla padre**, la geometría de las capas hijas de **puntos/líneas** y los **adjuntos** en las capas equivalentes del ítem destino.

El proceso es idempotente: utiliza el `globalid` de la geometría hija como `id_unique` en el destino. Primero se ejecuta siempre en simulación; la escritura requiere habilitar explícitamente `EXECUTE_CHANGES`.


In [ ]:
from pathlib import Path

from IPython.display import display
from Lib.esrilogs import Logfile, capturaError
from configuracion_ciren import load_solution_config, connect_gis_from_config
from consolidar_survey import run_consolidation, validate_schema


## 1. Configuración

Solo son elegibles los padres con `validacion = 'si'`. Sus `globalid` se cruzan con `parentglobalid` de puntos y líneas; después se excluyen las geometrías cuyo `globalid` ya está guardado como `id_unique` en el destino.


In [ ]:
CONFIG_PATH = Path('configuracion_ciren.json')
config = load_solution_config(str(CONFIG_PATH))
items = config['items']
consolidation = config['consolidation']
log_config = config['logs']

logs = Logfile(
    'ConsolidacionSurveyNotebook',
    log_path=Path(log_config['path']),
    max_age_days=log_config.get('max_age_days', 30),
    rotate_mode=log_config.get('rotate_mode', 'archive'),
)
logs.start_script('Inicio notebook de consolidacion')

SOURCE_ITEM_ID = items['survey_feature_service']
TARGET_ITEM_ID = items['target_feature_service']
UNIQUE_FIELD = consolidation['unique_field']
PARENT_WHERE = consolidation['parent_where']
UPDATE_EXISTING = consolidation['update_existing']
SYNC_ATTACHMENTS = consolidation['sync_attachments']
EXECUTE_CHANGES = consolidation['notebook_execute_changes']

## 2. Conexión y validación del esquema

Las credenciales se leen desde un archivo externo y no quedan almacenadas en el notebook. Esta etapa solo consulta metadatos.


In [ ]:
gis = connect_gis_from_config(config)

source_item = gis.content.get(SOURCE_ITEM_ID)
target_item = gis.content.get(TARGET_ITEM_ID)
assert source_item is not None, 'No se encontró el ítem Survey.'
assert target_item is not None, 'No se encontró el ítem destino.'

schema = validate_schema(source_item, target_item, UNIQUE_FIELD)
logs.info(f'Origen: {source_item.title}')
logs.info(f'Destino: {target_item.title}')
logs.info('Esquema validado correctamente')

### Mapeo de cotas durante la consolidación

| Geometría | Campo Survey | Significado | Campo destino |
|---|---|---|---|
| Punto | `cota` | Cota automática del GPS | `cota` |
| Punto | `cota_manual` | Cota ingresada manualmente | `cota_manual` |
| Línea | `cota` | Cota inicial ingresada manualmente | `cota_inicial` |
| Línea | `cota_manual` | Cota final ingresada manualmente | `cota_final` |

## 3. Diagnóstico de volúmenes

Permite confirmar el universo que se procesará antes de la simulación.


In [ ]:
diagnostic = {
    'padres_seleccionados': schema['source_table'].query(
        where=PARENT_WHERE, return_count_only=True
    ),
    'puntos_origen': schema['source_by_type']['esriGeometryPoint'].query(
        return_count_only=True
    ),
    'lineas_origen': schema['source_by_type']['esriGeometryPolyline'].query(
        return_count_only=True
    ),
    'puntos_destino': schema['target_by_type']['esriGeometryPoint'].query(
        return_count_only=True
    ),
    'lineas_destino': schema['target_by_type']['esriGeometryPolyline'].query(
        return_count_only=True
    ),
}
display(diagnostic)

## 4. Simulación obligatoria

No crea ni actualiza entidades. El reporte anticipa inserciones, registros ya cargados según `id_unique`, geometrías huérfanas y adjuntos por copiar.


In [ ]:
simulation_report = run_consolidation(
    gis=gis,
    source_item_id=SOURCE_ITEM_ID,
    target_item_id=TARGET_ITEM_ID,
    unique_field=UNIQUE_FIELD,
    parent_where=PARENT_WHERE,
    dry_run=True,
    update_existing=UPDATE_EXISTING,
    sync_attachments=SYNC_ATTACHMENTS,
    logs=logs,
)
display(simulation_report)
logs.info(f'Reporte de simulacion: {simulation_report}')

assert sum(v['failed'] for v in simulation_report.values()) == 0
assert sum(v['orphaned'] for v in simulation_report.values()) == 0

## 5. Ejecutar actualización

Revise el reporte anterior. Después cambie `consolidation.notebook_execute_changes` a `true` en `configuracion_ciren.json` y ejecute esta celda. No se eliminan entidades ni adjuntos.


In [ ]:
if not EXECUTE_CHANGES:
    print('Ejecución bloqueada: cambie EXECUTE_CHANGES a True después de revisar la simulación.')
else:
    execution_report = run_consolidation(
        gis=gis,
        source_item_id=SOURCE_ITEM_ID,
        target_item_id=TARGET_ITEM_ID,
        unique_field=UNIQUE_FIELD,
        parent_where=PARENT_WHERE,
        dry_run=False,
        update_existing=UPDATE_EXISTING,
        sync_attachments=SYNC_ATTACHMENTS,
        logs=logs,
    )
    display(execution_report)
    logs.info(f'Reporte de ejecucion: {execution_report}')

## 6. Verificación posterior

Ejecute después de una actualización real para confirmar los totales publicados.


In [ ]:
verification = {
    'puntos_destino': schema['target_by_type']['esriGeometryPoint'].query(
        return_count_only=True
    ),
    'lineas_destino': schema['target_by_type']['esriGeometryPolyline'].query(
        return_count_only=True
    ),
}
display(verification)
logs.end(f'Verificacion final: {verification}')
logs.close('Notebook de consolidacion finalizado')